## Import Libraries

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim, when, lower, upper, lit, count, regexp_replace, concat, substring, length, lpad

## Reading from bronze layer

In [0]:
df = spark.table("olist.bronze.geolocation")
df.display()

## Overview about the table

In [0]:
# Table info
print("=== Schema ===")
df.printSchema()

print("=== Row Count ===")
print(f"Total rows: {df.count()}")

print("=== Null Counts per Column ===")
df.select([
    F.count(F.when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).display()

print("=== Zip Code Length ==")
df.select(col("geolocation_zip_code_prefix"), length(col("geolocation_zip_code_prefix")).alias("zip_code_length")).distinct().display()

## Transformations

### 1. Trim all whitespaces from string columns

In [0]:
for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
    df = df.withColumn(field.name, trim(col(field.name)))
  

### 2. Normalize improperly represented nulls in string columns

In [0]:
NULL_STRINGS = ["", "null", "none", "n/a", "na", "unknown", "-"]

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(
            field.name,
            when(lower(trim(col(field.name))).isin(NULL_STRINGS), None)
            .otherwise(col(field.name))
        )

### 3. Standardize string columns

In [0]:
# Standardize geolocation_state to upper case
df = (df.withColumn("geolocation_state", upper(col("geolocation_state"))))

In [0]:
# Standardize geolocation_city 
df = df.withColumn(
    "geolocation_city",
    trim(regexp_replace(col("geolocation_city"), r"\s+", " ")) 
)

df = df.withColumn(
    "geolocation_city",
    concat(
        upper(substring(col("geolocation_city"), 1, 1)),    
        lower(substring(col("geolocation_city"), 2, 9999))  
    )
)

### 4. Normalize zip code

In [0]:
df = df.withColumn("geolocation_zip_code_prefix", lpad(col("geolocation_zip_code_prefix"), 5, "0"))


In [0]:
# Brazil roughly fits between latitude -34..6 and longitude -74..-34.
# This drops clearly wrong points (ocean, other countries).
df = df.filter((col("geolocation_lat") >= -34) & (col("geolocation_lat") <= 6))
df = df.filter((col("geolocation_lng") >= -74) & (col("geolocation_lng") <= -34))

### 5. Handle nulls

In [0]:
df = df.filter(
    col("geolocation_zip_code_prefix").isNotNull() &
    col("geolocation_lat").isNotNull() &
    col("geolocation_lng").isNotNull()
)

### 6. Remove duplicate rows 

In [0]:
df = df.dropDuplicates()

### 7. Quality Check

In [0]:
print(f"Total rows after cleaning: {df.count()}")
print(f"Distinct zip codes: {df.select('geolocation_zip_code_prefix').distinct().count()}")
print(f"Distinct states: {df.select('geolocation_state').distinct().count()}")

print("=== Null Counts per Column ===")
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).display()

print("=== Zip Code Length ===")
df.select(length(col("geolocation_zip_code_prefix")).alias("zip_code_length")).distinct().display()

df.display()

## Write to silver layer

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("olist.silver.geolocation")